In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 1: Setup (run after training)
import sys
sys.path.append(str('/content/drive/MyDrive/ResearchProject'))

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

from unet_denoiser import BlindVideoDenoiserUNet
from dataloader import BlindDenoiseDataset
from model_testing import ModelTester

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Cell 2: Load your best model
checkpoint_path = '/content/drive/MyDrive/ResearchProject/checkpoints/best_model.pt'

model = BlindVideoDenoiserUNet(in_channels=9, out_channels=3, base_channels=64, num_stages=3)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Model loaded from epoch {checkpoint['epoch']}")
print(f"Training loss: {checkpoint['train_loss']:.6f}")
print(f"Validation loss: {checkpoint['val_loss']:.6f}")

tester = ModelTester(model, device=device)

Model loaded from epoch 90
Training loss: 0.030602
Validation loss: 0.029864


In [4]:
# Cell 3: Test on a single video from DAVIS
davis_root = '/content/drive/MyDrive/ResearchProject/DAVISDataset'

# List available videos
videos = [v for v in os.listdir(davis_root) if os.path.isdir(os.path.join(davis_root, v))]
print(f"\nAvailable videos: {len(videos)}")
print("Videos:", videos[1:11])  # Show first 10

# Test on a specific video (change this to test different videos)
test_video = videos[5]  # Change index to test different videos
video_path = os.path.join(davis_root, test_video)

print(f"\n{'='*60}")
print(f"Testing on video: {test_video}")
print(f"{'='*60}")


Available videos: 150
Videos: ['weightlifting', 'water-slide', 'volleyball-beach', 'walking', 'varanus-cage', 'tuk-tuk', 'twist-dance', 'trucks-race', 'tram', 'tractor-sand']

Testing on video: varanus-cage


In [5]:
# Cell 4: Add noise and denoise (correct pipeline)
# This adds synthetic noise to clean frames, then denoises the NOISY input.
# All processing happens at the same resolution to avoid interpolation artifacts.

noise_std = 50  # Change this to test different noise levels (5-255)
process_res = (256, 256)  # Processing resolution (must be divisible by 8)

clean_frames, noisy_frames, denoised_frames, frame_names = tester.denoise_noisy_video(
    video_path,
    noise_std=noise_std,
    resize_to=process_res,
    seed=42
)

print(f"\nSuccessfully denoised {len(denoised_frames)} frames!")
print(f"Noise level: σ = {noise_std}")
print(f"Resolution: {process_res[1]}x{process_res[0]}")

Loaded 67 frames from varanus-cage
Processing resolution: 256x256
Noise σ = 50
Denoising noisy frames...
  20/67 frames done
  40/67 frames done
  60/67 frames done
  Done — 67 frames denoised

Successfully denoised 67 frames!
Noise level: σ = 50
Resolution: 256x256


In [6]:
# Cell 5: Quick sanity check — verify frames are consistent
print(f"Clean frames:    {len(clean_frames)}, shape {clean_frames[0].shape}, range [{clean_frames[0].min():.3f}, {clean_frames[0].max():.3f}]")
print(f"Noisy frames:    {len(noisy_frames)}, shape {noisy_frames[0].shape}, range [{noisy_frames[0].min():.3f}, {noisy_frames[0].max():.3f}]")
print(f"Denoised frames: {len(denoised_frames)}, shape {denoised_frames[0].shape}, range [{denoised_frames[0].min():.3f}, {denoised_frames[0].max():.3f}]")

# Quick per-frame PSNR check
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
sample_noisy_psnr = psnr_metric(
    (clean_frames[0] * 255).astype(np.uint8),
    (noisy_frames[0] * 255).astype(np.uint8),
    data_range=255
)
sample_denoised_psnr = psnr_metric(
    (clean_frames[0] * 255).astype(np.uint8),
    (denoised_frames[0] * 255).astype(np.uint8),
    data_range=255
)
print(f"\nFrame 0 — Noisy PSNR: {sample_noisy_psnr:.2f} dB, Denoised PSNR: {sample_denoised_psnr:.2f} dB")

Clean frames:    67, shape (256, 256, 3), range [0.000, 1.000]
Noisy frames:    67, shape (256, 256, 3), range [0.000, 1.000]
Denoised frames: 67, shape (256, 256, 3), range [0.000, 1.000]

Frame 0 — Noisy PSNR: 14.96 dB, Denoised PSNR: 25.49 dB


In [7]:
# Cell 6: Compute metrics
print(f"\n{'='*60}")
print(f"Metrics for {test_video} — noise σ = {noise_std}")
print(f"{'='*60}")

metrics = tester.compute_metrics(noisy_frames, clean_frames, denoised_frames)

print(f"\nNoisy Image PSNR:     {metrics['noisy_psnr_mean']:.2f} dB")
print(f"Denoised Image PSNR:  {metrics['denoised_psnr_mean']:.2f} dB")
print(f"PSNR Improvement:     +{metrics['psnr_improvement']:.2f} dB")
print()
print(f"Noisy Image SSIM:     {metrics['noisy_ssim_mean']:.4f}")
print(f"Denoised Image SSIM:  {metrics['denoised_ssim_mean']:.4f}")
print(f"SSIM Improvement:     +{metrics['ssim_improvement']:.4f}")


Metrics for varanus-cage — noise σ = 50

Noisy Image PSNR:     15.11 dB
Denoised Image PSNR:  25.48 dB
PSNR Improvement:     +10.37 dB

Noisy Image SSIM:     0.2818
Denoised Image SSIM:  0.7721
SSIM Improvement:     +0.4903


In [8]:
# Cell 7: Visualize results on frames from the test video
print(f"\n{'='*60}")
print(f"Visualizing results for {test_video} (σ={noise_std})")
print(f"{'='*60}")

# Show up to 4 evenly-spaced frames
num_frames_to_show = min(4, len(denoised_frames))
frame_step = max(1, len(denoised_frames) // num_frames_to_show)

for frame_idx in range(0, len(denoised_frames), frame_step):
    if frame_idx >= len(denoised_frames):
        break
    print(f"  Frame {frame_idx + 1}/{len(denoised_frames)}")
    tester.visualize_results(
        clean_frames[frame_idx],
        noisy_frames[frame_idx],
        denoised_frames[frame_idx]
    )

print(f"\n{'='*60}")
print(f"Visualization complete!")
print(f"{'='*60}")

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# Cell 8: Save denoised video frames (optional)
output_dir = '/content/drive/MyDrive/ResearchProject/denoised_output'
os.makedirs(output_dir, exist_ok=True)

output_video_dir = os.path.join(output_dir, f"{test_video}_sigma{noise_std}")
os.makedirs(output_video_dir, exist_ok=True)

for frame_name, denoised_frame in zip(frame_names, denoised_frames):
    output_img = Image.fromarray((denoised_frame * 255).astype(np.uint8))
    output_path = os.path.join(output_video_dir, frame_name)
    output_img.save(output_path)

print(f"\nDenoised frames saved to: {output_video_dir}")


Denoised frames saved to: /content/drive/MyDrive/ResearchProject/denoised_output/varanus-cage_sigma50


In [10]:
# Cell 9: Test on multiple videos and compare
print(f"\n{'='*60}")
print(f"Testing on multiple videos (σ={noise_std})...")
print(f"{'='*60}")

results = []

for video_idx, video_name in enumerate(videos[:5]):  # Test on first 5 videos
    v_path = os.path.join(davis_root, video_name)

    try:
        clean, noisy, denoised, _ = tester.denoise_noisy_video(
            v_path,
            noise_std=noise_std,
            resize_to=process_res,
            seed=42 + video_idx  # Different seed per video
        )

        m = tester.compute_metrics(noisy, clean, denoised)
        results.append({
            'video': video_name,
            'denoised_psnr': m['denoised_psnr_mean'],
            'psnr_improvement': m['psnr_improvement'],
            'denoised_ssim': m['denoised_ssim_mean'],
            'ssim_improvement': m['ssim_improvement']
        })

        print(f"  {video_name}: PSNR={m['denoised_psnr_mean']:.2f} dB (+{m['psnr_improvement']:.2f}), "
              f"SSIM={m['denoised_ssim_mean']:.4f} (+{m['ssim_improvement']:.4f})")

    except Exception as e:
        print(f"  {video_name}: Error - {str(e)}")


Testing on multiple videos (σ=50)...
Loaded 65 frames from upside-down
Processing resolution: 256x256
Noise σ = 50
Denoising noisy frames...
  20/65 frames done
  40/65 frames done
  60/65 frames done
  Done — 65 frames denoised
  upside-down: PSNR=25.04 dB (+9.74), SSIM=0.7903 (+0.4897)
Loaded 90 frames from weightlifting
Processing resolution: 256x256
Noise σ = 50
Denoising noisy frames...
  20/90 frames done
  40/90 frames done
  60/90 frames done
  80/90 frames done
  Done — 90 frames denoised
  weightlifting: PSNR=26.34 dB (+10.96), SSIM=0.8146 (+0.5307)
Loaded 88 frames from water-slide
Processing resolution: 256x256
Noise σ = 50
Denoising noisy frames...
  20/88 frames done
  40/88 frames done
  60/88 frames done
  80/88 frames done
  Done — 88 frames denoised
  water-slide: PSNR=26.08 dB (+10.87), SSIM=0.7864 (+0.5517)
Loaded 73 frames from volleyball-beach
Processing resolution: 256x256
Noise σ = 50
Denoising noisy frames...
  20/73 frames done
  40/73 frames done
  60/73 fra

In [11]:
# Cell 10: Summary statistics
if results:
    print(f"\n{'='*60}")
    print(f"Summary Statistics (across {len(results)} videos):")
    print(f"{'='*60}")

    psnr_improvements = [r['psnr_improvement'] for r in results]
    ssim_improvements = [r['ssim_improvement'] for r in results]

    print(f"Average PSNR Improvement: +{np.mean(psnr_improvements):.2f} dB")
    print(f"Std Dev PSNR:             {np.std(psnr_improvements):.2f} dB")
    print(f"Min/Max PSNR:             {np.min(psnr_improvements):.2f} / {np.max(psnr_improvements):.2f} dB")
    print()
    print(f"Average SSIM Improvement: +{np.mean(ssim_improvements):.4f}")
    print(f"Std Dev SSIM:             {np.std(ssim_improvements):.4f}")
    print(f"Min/Max SSIM:             {np.min(ssim_improvements):.4f} / {np.max(ssim_improvements):.4f}")


Summary Statistics (across 5 videos):
Average PSNR Improvement: +11.42 dB
Std Dev PSNR:             1.59 dB
Min/Max PSNR:             9.74 / 14.45 dB

Average SSIM Improvement: +0.5499
Std Dev SSIM:             0.0530
Min/Max SSIM:             0.4897 / 0.6480
